# Векторні операції над словами (word embeddings)

Імпорти. Модель беремо готову через gensim (glove), бо вона вже навчена на великому тексті.

In [ ]:
import numpy as np
import pandas as pd
import gensim.downloader as api
from sklearn.decomposition import PCA

### 1. DataFrame з тривимірними векторами слів

Завантажуємо модель (це файл з embeddings, качається з інтернету). Беремо вектори розмірності 50.

In [ ]:
model = api.load('glove-wiki-gigaword-50')

Не будемо брати всі слова (їх дуже багато), а виберемо свій список слів з якими буде зручно експериментувати.

In [ ]:
words = ['king', 'queen', 'man', 'woman', 'boy', 'girl', 'prince', 'princess',
         'paris', 'france', 'london', 'england', 'rome', 'italy', 'berlin', 'germany',
         'dog', 'cat', 'horse', 'lion', 'apple', 'orange', 'water', 'fire', 'sun', 'moon',
         'day', 'night', 'one', 'two', 'three', 'car', 'road', 'book', 'school', 'music']

vectors = np.array([model[w] for w in words])
vectors.shape

Вектори у нас 50-вимірні, а нам треба 3 виміри. Зменшуємо розмірність через PCA.

In [ ]:
pca = PCA(n_components=3)
vectors_3d = pca.fit_transform(vectors)
vectors_3d.shape

In [ ]:
df = pd.DataFrame(vectors_3d, columns=['x', 'y', 'z'])
df.insert(0, 'word', words)
df.head(10)

### 2. Функція пошуку найближчого слова

Функція приймає 3D вектор і шукає в нашому DataFrame слово, яке найближче до нього (по евклідовій відстані).

In [ ]:
def nearest_word(vec):
    coords = df[['x', 'y', 'z']].values
    dist = np.linalg.norm(coords - vec, axis=1)
    return df.iloc[np.argmin(dist)]['word']

def get_vec(word):
    return df[df['word'] == word][['x', 'y', 'z']].values[0]

Перевіримо: якщо подати вектор самого слова, має повернутись це саме слово.

In [ ]:
print(nearest_word(get_vec('king')))
print(nearest_word(get_vec('dog')))
print(nearest_word(get_vec('paris')))

Все правильно, функція знаходить те саме слово.

### 3. Векторний добуток (ортогональне слово)

Беремо пари слів, рахуємо їх векторний добуток (np.cross) - це вектор перпендикулярний до обох. Потім дивимось яке слово до нього найближче.

In [ ]:
pairs = [('king', 'queen'), ('paris', 'france'), ('day', 'night'), ('dog', 'cat'), ('sun', 'moon')]

for a, b in pairs:
    c = np.cross(get_vec(a), get_vec(b))
    print(a, 'x', b, '->', nearest_word(c))

Видно що слово яке найближче до векторного добутку не пов'язане за змістом з вихідною парою. Це логічно, бо векторний добуток дає вектор перпендикулярний (ортогональний) до обох слів, а ортогональні вектори у embeddings означають що слова не схожі між собою.

### 4. Кут між словами

Кут рахуємо через косинус: cos = (a·b)/(|a|*|b|), а потім arccos. Чим менший кут - тим слова більш схожі.

In [ ]:
def angle_between(a, b):
    va, vb = get_vec(a), get_vec(b)
    cos = np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb))
    cos = np.clip(cos, -1, 1)
    return np.degrees(np.arccos(cos))

In [ ]:
test_pairs = [('king', 'queen'), ('man', 'woman'), ('paris', 'london'),
              ('dog', 'cat'), ('day', 'night'), ('king', 'dog')]

for a, b in test_pairs:
    print(a, '-', b, ':', round(angle_between(a, b), 1), 'градусів')

По кутах добре видно зв'язок між словами. Схожі за змістом слова (king-queen, man-woman, paris-london, dog-cat) мають маленький кут (десь 13-20 градусів), а не пов'язані слова (king-dog) мають великий кут (близько 87 градусів, тобто майже перпендикулярні).

### Висновок

В цій роботі я працювала з векторами слів (word embeddings) з моделі glove.

Спочатку завантажила модель, вибрала список слів і витягнула їх вектори. Вектори були 50-вимірні, тому через PCA я зменшила їх до 3 вимірів і склала в DataFrame.

Потім написала функцію яка по вектору знаходить найближче слово і перевірила що вона працює правильно.

Далі рахувала векторний добуток пар слів - виявилось що найближче до нього слово не пов'язане з парою, бо векторний добуток дає перпендикулярний вектор.

В кінці написала функцію для кута між словами. По результатах видно що схожі слова мають маленький кут, а різні за змістом - великий. Тобто кут між векторами добре показує наскільки слова близькі за значенням.